# Notebook 2 - Training Dua Metode Deep Learning
## Metode 1: Deep MLP  |  Metode 2: TabNet (Keras, TFLite-friendly)

Kedua model dibangun murni di `tf.keras` agar dapat diekspor ke **TensorFlow Lite**
untuk inferensi *on-device* di aplikasi Flutter. Imbalance ditangani dengan **class weight**.

In [ ]:
import numpy as np, tensorflow as tf, json
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
d = np.load("../outputs/processed.npz")
Xtr,Xva,Xte,ytr,yva,yte = d["Xtr"],d["Xva"],d["Xte"],d["ytr"],d["yva"],d["yte"]
INPUT_DIM = Xtr.shape[1]
cw = compute_class_weight("balanced", classes=np.array([0,1]), y=ytr)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print("INPUT_DIM:", INPUT_DIM, "| class_weight:", class_weight)

### Metode 1 - Deep MLP
Jaringan feedforward: 3 hidden layer (128-64-32) + BatchNorm + Dropout 0.3.

In [ ]:
def build_mlp(input_dim, l2=1e-4):
    inp = Input(shape=(input_dim,), name="features")
    x = layers.BatchNormalization()(inp)
    for u in (128,64,32):
        x = layers.Dense(u, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
        x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation="sigmoid", name="risk")(x)
    return Model(inp, out, name="MLP")
build_mlp(INPUT_DIM).summary()

### Metode 2 - TabNet
Mempertahankan ide inti TabNet: pemrosesan **multi-step** berurutan, **attentive
transformer** (mask seleksi fitur), dan **feature transformer** berbasis GLU. Mask memakai
softmax (bukan sparsemax) agar 100% kompatibel dengan TFLite untuk deployment mobile.

In [ ]:
def glu(x, u):
    return layers.Multiply()([layers.Dense(u)(x), layers.Dense(u, activation="sigmoid")(x)])

def build_tabnet(input_dim, feature_dim=32, n_steps=3):
    inp = Input(shape=(input_dim,), name="features")
    x = layers.BatchNormalization()(inp)
    prior = layers.Lambda(lambda t: tf.ones_like(t))(x)
    agg = None
    for s in range(n_steps):
        att = layers.BatchNormalization()(layers.Dense(input_dim)(x))
        att = layers.Multiply()([att, prior])
        mask = layers.Softmax(name=f"mask_{s}")(att)
        prior = layers.Multiply()([prior, layers.Lambda(lambda m: 1.5-m)(mask)])
        masked = layers.Multiply()([inp, mask])
        ft = layers.BatchNormalization()(glu(masked, feature_dim)); ft = glu(ft, feature_dim)
        dec = layers.Activation("relu")(ft)
        agg = dec if agg is None else layers.Add()([agg, dec])
        x = layers.Dense(feature_dim, activation="relu")(ft)
    out = layers.Dense(1, activation="sigmoid", name="risk")(agg)
    return Model(inp, out, name="TabNet")
build_tabnet(INPUT_DIM).summary()

In [ ]:
import matplotlib.pyplot as plt

### Training kedua model

In [ ]:
def train(builder, name):
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    m = builder(INPUT_DIM)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    es = EarlyStopping(monitor="val_auc", mode="max", patience=10, restore_best_weights=True)
    h = m.fit(Xtr,ytr, validation_data=(Xva,yva), epochs=80, batch_size=512,
              class_weight=class_weight, callbacks=[es], verbose=0)
    m.save(f"../outputs/models/{name}.keras")
    np.save(f"../outputs/models/hist_{name}.npy", h.history)
    print(f"{name}: {len(h.history['loss'])} epoch, best val_auc={max(h.history['val_auc']):.4f}")
    return h

for b,n in [(build_mlp,"MLP"),(build_tabnet,"TabNet")]:
    h = train(b,n)
    fig, ax = plt.subplots(1,2, figsize=(11,4))
    ax[0].plot(h.history["accuracy"],label="Train"); ax[0].plot(h.history["val_accuracy"],label="Val")
    ax[0].set_title(f"Accuracy - {n}"); ax[0].legend()
    ax[1].plot(h.history["loss"],label="Train"); ax[1].plot(h.history["val_loss"],label="Val")
    ax[1].set_title(f"Loss - {n}"); ax[1].legend(); plt.tight_layout(); plt.show()